# TabPFN variants

Every other model adds nothing to the TabPFN + GP stack (out-of-fold 0.7800): SVR, XGBoost, Extra Trees, MLP and Lasso all change it by less than ±0.0005. They all see the same 216 kNN-filled features, and their errors are 0.79–0.88 correlated with TabPFN's.

So this notebook gives TabPFN **different views of the data**, each a candidate column for the stack:

| Variant | Rows | Features | Missing values |
|---|---|---|---|
| `tabpfn` (reference, `TabPFN.ipynb`) | 1152 | the 216 selected, log on 11 | kNN fill |
| `tabpfn_nan` | 1152 | the 216 selected, log on 11 | **left as NaN** (TabPFN handles them) |
| `tabpfn_nan_300` | 1152 | **top 300** by \|Spearman ρ\| | left as NaN |
| `tabpfn_nan_all` | 1152 | **all 828** non-constant | left as NaN |
| `tabpfn_seeds` | 1152 | as the reference | kNN fill, **average of several `random_state`** |

Same 1152 rows (outliers removed), same `y`, same shared folds as every other model.

**Order:** screening with one seed (cell 4) → pick what helps the stack → final out-of-fold and test files with several seeds (cell 5) → one line in `MODELS` of `Stacking/stacking.ipynb`.

In [ ]:
## Data

import os
import time
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from tabpfn import TabPFNRegressor

os.environ["TABPFN_TOKEN"] = "PASTE_YOUR_KEY_HERE"   # ⚠️ put "PASTE_YOUR_KEY_HERE" back before any commit

X_knn = pd.read_csv("../X_train_log_knn.csv", index_col="id")          # the reference input: 1152 rows × 216
X_test_knn = pd.read_csv("../X_test_log_knn.csv", index_col="id")
y = pd.read_csv("../y_train.csv", index_col="id").loc[X_knn.index, "y"]
fold = pd.read_csv("../Stacking/folds.csv", index_col="id").loc[X_knn.index, "fold"]

# raw data, NaNs kept: same 1152 rows, constant columns dropped (as in data_prep.ipynb)
CONSTANT = ["x104", "x129", "x489", "x530"]
LOG_COLS = ["x115", "x159", "x415", "x458", "x507", "x531", "x542", "x641", "x665", "x702", "x742"]   # from data_prep.ipynb
raw = pd.read_csv("../X_train.csv", index_col="id").loc[X_knn.index].drop(columns=CONSTANT)
raw_test = pd.read_csv("../X_test.csv", index_col="id").drop(columns=CONSTANT)
raw[LOG_COLS], raw_test[LOG_COLS] = np.log(raw[LOG_COLS]), np.log(raw_test[LOG_COLS])   # log(NaN) stays NaN

print(f"knn {X_knn.shape} | raw {raw.shape} (NaN {raw.isna().mean().mean():.1%}) | test {raw_test.shape}")

/opt/anaconda3/envs/AML/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


knn (1152, 216) | raw (1152, 828) (NaN 7.6%) | test (776, 828)


In [3]:
## Feature sets

# |Spearman ρ| of every feature with the age, NaNs skipped (the same ranking as the 216-feature selection)
rho = raw.apply(lambda col: col.corr(y, method="spearman")).abs().sort_values(ascending=False)

SELECTED = list(X_knn.columns)
VARIANTS = {   # name -> (training features, test features); the heaviest last
    "tabpfn_nan":     (raw[SELECTED], raw_test[SELECTED]),
    "tabpfn_seeds":   (X_knn, X_test_knn),
    "tabpfn_nan_300": (raw[rho.index[:300]], raw_test[rho.index[:300]]),
    "tabpfn_nan_all": (raw, raw_test),
}
for name, (X, X_test) in VARIANTS.items():
    print(f"{name:15s}: {X.shape[1]} features, NaN {X.isna().mean().mean():.1%}")
print(f"the 216 selected = the top 216 by |ρ|: {set(SELECTED) == set(rho.index[:216])}")

tabpfn_nan     : 216 features, NaN 7.6%
tabpfn_seeds   : 216 features, NaN 0.0%
tabpfn_nan_300 : 300 features, NaN 7.6%
tabpfn_nan_all : 828 features, NaN 7.6%
the 216 selected = the top 216 by |ρ|: True


In [4]:
## Helpers

DEVICE = "cpu"   # the M1's GPU (MPS) runs out of memory; on a Mac with more memory, "auto" is much faster

def tabpfn_predict(X_train, y_train, X_pred, seeds):
    # average of one TabPFN per random_state
    return np.mean([TabPFNRegressor(random_state=s, device=DEVICE).fit(X_train, y_train).predict(X_pred) for s in seeds], axis=0)

def out_of_fold(X, seeds):
    oof = pd.Series(np.nan, index=X.index)
    for f in range(5):
        train, val = fold != f, fold == f
        oof[val] = tabpfn_predict(X[train], y[train], X[val], seeds)
        print(f"  fold {f} done", flush=True)
    return oof

# the current stack's out-of-fold predictions
stack_oof = pd.concat({m: pd.read_csv(f"../Stacking/oof_{m}.csv", index_col="id").iloc[:, 0]
                       for m in ["tabpfn", "gp"]}, axis=1).loc[X_knn.index]

def stack_r2(columns):
    # linear stacking scored on the shared folds, as in Stacking/stacking.ipynb
    pred = pd.Series(np.nan, index=columns.index)
    for f in range(5):
        train, val = fold != f, fold == f
        pred[val] = LinearRegression().fit(columns[train], y[train]).predict(columns[val])
    return r2_score(y, pred)

print(f"reference: TabPFN alone {r2_score(y, stack_oof['tabpfn']):.4f} | stack TabPFN + GP {stack_r2(stack_oof):.4f}")

reference: TabPFN alone 0.7772 | stack TabPFN + GP 0.7800


In [5]:
## Screening: one seed per variant (the seeds variant uses 3)

screen = {}
for name, (X, X_test) in VARIANTS.items():
    start = time.time()
    oof = out_of_fold(X, seeds=[0, 1, 2] if name == "tabpfn_seeds" else [0])
    screen[name] = oof
    candidates = stack_oof.assign(**{name: oof})
    print(f"{name:15s}: alone {r2_score(y, oof):.4f} | error corr. with TabPFN {(oof - y).corr(stack_oof['tabpfn'] - y):.3f} | "
          f"stack TabPFN + GP + it {stack_r2(candidates):.4f} | ({time.time() - start:.0f} s)")

# the seeds variant is a replacement for the TabPFN column, not an extra one
print(f"\nstack with tabpfn_seeds instead of tabpfn: {stack_r2(stack_oof.assign(tabpfn=screen['tabpfn_seeds'])):.4f}")

  fold 0 done
  fold 1 done
  fold 2 done
  fold 3 done
  fold 4 done
tabpfn_nan     : alone 0.7784 | error corr. with TabPFN 0.985 | stack TabPFN + GP + it 0.7823 | (387 s)
  fold 0 done
  fold 1 done
  fold 2 done
  fold 3 done
  fold 4 done
tabpfn_seeds   : alone 0.7783 | error corr. with TabPFN 0.997 | stack TabPFN + GP + it 0.7801 | (3966 s)
  fold 0 done
  fold 1 done
  fold 2 done
  fold 3 done
  fold 4 done
tabpfn_nan_300 : alone 0.7796 | error corr. with TabPFN 0.976 | stack TabPFN + GP + it 0.7825 | (510 s)
  fold 0 done
  fold 1 done
  fold 2 done
  fold 3 done
  fold 4 done
tabpfn_nan_all : alone 0.7667 | error corr. with TabPFN 0.973 | stack TabPFN + GP + it 0.7791 | (1645 s)

stack with tabpfn_seeds instead of tabpfn: 0.7807


**How to read it:** the reference stack (TabPFN + GP) is 0.7800. A variant is worth keeping if the stack with it gains more than about 0.002; differences below that are noise. A low error correlation with TabPFN is what makes a variant useful.

Then set `KEEP` below to the variants that helped, and run the last cell.

In [6]:
## Final files for the kept variants: out-of-fold + test predictions, average of SEEDS

KEEP = ["tabpfn_nan_300"]      # chosen from the screening: nan_300 + GP is the best stack (0.7829)
SEEDS = [0]                    # extra seeds did not help (tabpfn_seeds: +0.0001)

for name in KEEP:
    X, X_test = VARIANTS[name]
    oof = out_of_fold(X, SEEDS).rename(name)
    oof.to_csv(f"../Stacking/oof_{name}.csv")
    pd.Series(tabpfn_predict(X, y, X_test, SEEDS), index=X_test.index, name="y").to_csv(f"submission_{name}.csv")
    print(f"{name}: out-of-fold R2 {r2_score(y, oof):.4f} -> add to MODELS: "
          f'"{name}": ROOT / "TabPFN" / "submission_{name}.csv",')

KeyboardInterrupt: 

In [7]:
## Combinations of the useful variants (no new TabPFN runs)

combos = {
    "TabPFN + GP (current)":                    stack_oof,
    "+ nan":                                    stack_oof.assign(tabpfn_nan=screen["tabpfn_nan"]),
    "+ nan_300":                                stack_oof.assign(tabpfn_nan_300=screen["tabpfn_nan_300"]),
    "+ nan + nan_300":                          stack_oof.assign(tabpfn_nan=screen["tabpfn_nan"], tabpfn_nan_300=screen["tabpfn_nan_300"]),
    "nan_300 + GP (replacing TabPFN)":          stack_oof.drop(columns="tabpfn").assign(tabpfn_nan_300=screen["tabpfn_nan_300"]),
    "nan + nan_300 + GP (replacing TabPFN)":    stack_oof.drop(columns="tabpfn").assign(tabpfn_nan=screen["tabpfn_nan"], tabpfn_nan_300=screen["tabpfn_nan_300"]),
}
for name, cols in combos.items():
    print(f"{name:40s}: stack R2 {stack_r2(cols):.4f}")


TabPFN + GP (current)                   : stack R2 0.7800
+ nan                                   : stack R2 0.7823
+ nan_300                               : stack R2 0.7825
+ nan + nan_300                         : stack R2 0.7817
nan_300 + GP (replacing TabPFN)         : stack R2 0.7829
nan + nan_300 + GP (replacing TabPFN)   : stack R2 0.7816
